#BINFA TP5 : modélisation de l'activité électrique de neurones

##Exercice 1 : modèle de Hodgkin-Huxley

Implémentez le système d'équation différentielles d'Hodgkin-Huxley:


$$\left\{\begin{array}{lll}
C_m \frac{dV_m(t)}{dt} & = & I(t) - \bar{g}_{K} n(t)^4(V_m(t) - E_{K}) - \bar{g}_{Na} m(t)^3h(t)(V_m(t) - E_{Na}) - \bar{g}_l (V_m(t) - E_l)\\
\frac{dn(t)}{dt} & = & \alpha_n(V_m(t))(1 - n(t)) - \beta_n(V_m(t))n(t)\\
\frac{dm(t)}{dt} & = & \alpha_m(V_m(t))(1 - m(t)) - \beta_m(V_m(t))m(t)\\
\frac{dh(t)}{dt} & = & \alpha_h(V_m(t))(1 - h(t)) - \beta_h(V_m(t))h(t)
\end{array}\right .$$
et
$$\begin{array}{ll}
\alpha_n(V) = \frac{0.01(V + 55)}{1 - e^{\frac{-V - 55}{10}}} &
\beta_n(V) = 0.125e^{\frac{-V-65}{80}}\\
\alpha_m(V) = \frac{0.1(V + 40)}{1 - e^{\frac{-V -40}{10}}} &
\beta_m(V) = 4e^{\frac{-V-65}{18}}\\
\alpha_h(V) = 0.07e^{\frac{-V-65}{20}} &
\beta_h(V) = \frac{1}{1 + e^{\frac{-V-35}{10}}}
\end{array}$$

Pour la simulation, utilisez le schéma RK45 via la fonction *solve_ivp* de *scipy.integrate*.

Dans un premier temps, pour tester votre code, utilisez les paramètres suivants:
$Cm=1$, $gNa= 100$, $gK=5$, $gl=0.3$, $ENa=115$, $EK=-35$, $El=10.6$, $∆t=0.001$ sec (paramètre max_step de la fonction), $t_{max}=100$ sec. Pour valeurs initiales on utilisera $v_0=-65$ mV et on initialisera les autres variables à leurs valeurs d'équilibre:
$$m_0 = \frac{\alpha_m({v_0})}{\alpha_m(v_0) + \beta_m(v_0)}$$
$$n_0 = \frac{\alpha_n({v_0})}{\alpha_n(v_0) + \beta_n(v_0)}$$
$$h_0 = \frac{\alpha_h({v_0})}{\alpha_h(v_0) + \beta_h(v_0)}$$

Finalement, on injectera un courant constant $I(t)=7$.

In [ ]:
print("Votre code ici !!")

Pour comprendre comment se forme un potentiel d'action, affichez deux graphes l'un au dessus de l'autre : en haut le graphe de $V_m$ en fonction du temps et en dessous les valeurs de $m,n,h$ en fonction du temps.

In [ ]:
print("Votre code ici !!")

Basé sur ces graphes, expliquez comment se forme un potentiel d'action.

Votre réponse ici.

Testez le même modèle avec différentes valeurs de courant injecté $I(t) = k$ pour $k=\{0,5,10,15,20\}$. Qu'observez-vous?

In [ ]:
print("Votre code ici !!")

Pour quantifier l'effet, calculez le "spiking rate" : le nombre de potentiels d'actions émis divisé par le temps de simulation. Affichez un graphe du spiking rate en fonction de $k$.

In [ ]:
print("Votre code ici !!")

## Exercice 2 : Modèle Leaky Integrate and Fire (LIF)

Le modèle de Hodgkin-Huxley est assez précis mais très complexe. De nombreux autres modèles ont été dérivé pour reproduire différents aspects de la dynamique des neurones de manière simplifiée.

Le modèle LIF est l'un deux. Sa formulation est la suivante :
$$\left\{
\begin{array}{l}
\tau \frac{du(t)}{dt}=-u(t)+u_{rest}+\frac{I(t)}{gL}\\
\textrm{Si } u(t)≥u_{th} \textrm{ alors } u(t)= u_{th} \textrm{ et } u(t+k\Delta t)=u_{rest} \textrm{ tant que } k\Delta t <N_t\\
\end{array}\right .$$


Où $u(t)$ est le potentiel de membrane (en millivolts) du neurone au temps $t$, $u_{th}$ est le potentiel de seuil (en millivolts) au-delà duquel le neurone émet un potentiel d’action, u_rest est le potentiel de repos du neurone (en millivolts), $g_L$ est la conductance de fuite (en milli Siemens), $I(t)$ est le courant reçu par le neurone (en ampères) au temps $t$, $\tau$ est la constante de temps de membrane (en ms) et $N$ est la durée de la période réfractaire (en nombre de pas de temps $\Delta t$). Dans ce modèle, quand u dépasse la valeur seuil $u_{th}$ alors la valeur de u est forcée à $u_{th}$ au temps où cela se produit et à $u_{rest}$ pour les $N$ pas de temps suivants.

### Dynamique sous le seuil
Pour cette section, on se restreint au cas où $u(t) < u_{th}$ et on considère donc uniquement la première équation. Dans le cas ou $I(t)=I$ est une constante, ici on prend $I=0$, cette équation peut se résoudre analytiquement (assez simplement) et on obtient :

$$u(t)= u_{rest}+ \Delta u  e^{-t/τ},$$

où $u_{rest}$ est le potentiel de repos en mV et $\Delta u$ la différence de potentiel par rapport au potentiel de repos.

Affichez la courbe de $u$ en fonction de t pour $t=0\ldots tmax$ avec les paramètres suivants : $\tau=10$ ms, $u_{rest}=-65$ mV, $g_L=10$ mS, $\Delta t=$1e-4 s, $\Delta u=5$ mV, $t_{max}=0.1$ sec.

In [ ]:
print("Votre code ici !!")

### Simulation de la dynamique générale
On se place maintenant dans le cas général où on considère les deux équations (données au début de la section). La seconde équation introduisant une discontinuité on ne peut plus résoudre analytiquement le système. De la même manière, utiliser un solveur d'équations différentielle est problématique à cause de la discontinuité. A la place, on va le simuler en utilisant la méthode d'Euler à la main.

On considère les paramètres suivants : $I$ une fonction du temps représentant un courant injecté, $u0$ le potentiel initial de membrane (en mV), $dt$ le pas de temps de la simulation (en sec) et $tmax$ le temps maximal de simulation (en sec). Et les valeurs : $\tau=10$ ms, $u_{th}=-55$ mV, $u_{ref}=-65$ mV, $g_L=10$ mS, $N_t=30$ ms, $\Delta t=$1e-4 s, $u0=-60$ mV, $tmax=1$ sec.

Simulez le système et affichez le graphe de $u$ en fonction de $t$ pour $t=0..tmax.


In [ ]:
print("Votre code ici !!")

### Simulation stochastique
On va maintenant ajouter une composante aléatoire au modèle pour apporter de la variabilité au comportement du neurone. Votre fonction doit simuler le système suivant avec l'algorithme de Euler-Maruyama :

$$\left\{\begin{array}\tau \frac{du(t)}{dt}=-u(t)+u_{rest}+\frac{I(t)}/{g_L}   +\epsilon\eta\\
\textrm{Si } u(t)≥u_{th} \textrm{ alors } u(t)= u_{th} \textrm{ et } u(t+k\Delta t)=u_{rest} \textrm{ tant que } k\Delta t<N_t\\
\end{array}\right .$$

Où $\epsilon$ est l'amplitude du bruit (en mV) et $\eta \sim N(0,1)$ est une variable aléatoire distribuée selon une loi normale de moyenne 0 et variance 1.

On considère les paramètres suivants : $I$ une fonction du temps représentant un courant injecté, $v0$ le potentiel initial de membrane (en mV), $dt$ le pas de temps de la simulation (en sec) et $tmax$ le temps maximal de simulation (en sec). On considère les valeurs des paramètres suivants : $\tau =10$ ms, $u_{th}=-55$ mV, $u_{ref}=-65$ mV, $g_L=10$ mS, $N_t=30$ ms, $dt=$ 1e-4 s, $u0=-60$ mV, $\epsilon=10$ mV, tmax=1 sec.

Pour assurer la reproductibilité de vos résultats, vous devez réinitialiser le générateur aléatoire au début de votre simulation:

In [ ]:
import numpy as np
np.random.seed(42)

In [ ]:
print("Votre code ici !!")

### Détection des potentiels d'action

Chaque fois que le potentiel $u$ dépasse $u_{th}$ on a ce qu'on appelle un potentiel d'action (PA). De la même manière que pour Hodgkin-Huxley, on veut détecter la fréquence d'apparition de ces PA.

A partir du résultat d'une simulation: $ts$ un tableau des temps simulés et $u$ les valeurs du potentiel de membrane correspondant, détectez les temps auxquels des PA sont émis.


In [ ]:
print("Votre code ici !!")